In [25]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

df = pd.read_excel("./Diabetes (1).xlsx")
def create_lag_features(df, lags=2):
    df_lag = df.copy()
    for lag in range(1, lags + 1):
        df_lag[f"CPeptide_lag{lag}"] = df_lag.groupby("PatientID")["CPeptide"].shift(lag)
    return df_lag.dropna()

df_lagged = create_lag_features(df)

X_rf = df_lagged[['PatientID', 'TimeIndex', 'TimeMinutes', 'CPeptide', 'CPeptide_lag1', 'CPeptide_lag2']]
y_rf = df_lagged["ISR"]

X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(X_rf, y_rf, test_size=0.2, random_state=42)
X_train_rf

randomForrestmodel = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

randomForrestmodel.fit(X_train_rf, y_train_rf)
rf_pred = randomForrestmodel.predict(X_test_rf)

print("Random Forest RMSE:",
      np.sqrt(mean_squared_error(y_test_rf, rf_pred)))

Random Forest RMSE: 0.10594665564655774


In [14]:
df_lagged["ISR"].min(), df_lagged["ISR"].max()


(np.float64(-0.352473695883384), np.float64(1.340563192167483))

In [3]:
df_lagged

,PatientID,TimeIndex,TimeMinutes,CPeptide,ISR,CPeptide_lag1,CPeptide_lag2,CPeptide_lag3
3,1,3,18.947368,0.538470,0.661545,0.658418,0.946108,1.085871
4,1,4,25.263158,0.304155,0.431204,0.538470,0.658418,0.946108
5,1,5,31.578947,0.264084,0.146561,0.304155,0.538470,0.658418
6,1,6,37.894737,0.217787,0.327664,0.264084,0.304155,0.538470
7,1,7,44.210526,0.085067,0.154412,0.217787,0.264084,0.304155
...,...,...,...,...,...,...,...,...
1995,100,15,94.736842,0.065153,0.168655,0.131805,0.148872,-0.024851
1996,100,16,101.052632,0.068463,0.080532,0.065153,0.131805,0.148872
1997,100,17,107.368421,0.063921,0.176158,0.068463,0.065153,0.131805
1998,100,18,113.684211,0.108927,-0.050259,0.063921,0.068463,0.065153


In [4]:
df_lagged.columns.unique()

Index(['PatientID', 'TimeIndex', 'TimeMinutes', 'CPeptide', 'ISR',
       'CPeptide_lag1', 'CPeptide_lag2', 'CPeptide_lag3'],
      dtype='object')

In [20]:
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

xgb_model.fit(X_train_rf, y_train_rf)
xgb_pred = xgb_model.predict(X_test_rf)

print("XGBoost RMSE:",
      np.sqrt(mean_squared_error(y_test_rf, xgb_pred)))

XGBoost RMSE: 0.11085268368233894
